# Notebook 3: Transfer Learning with EfficientNet

## Overview

This notebook demonstrates **transfer learning** - the technique that takes our accuracy from ~62% (baseline CNN) to **99%+**:

1. **What is Transfer Learning?** - Using pre-trained ImageNet models as feature extractors
2. **Preprocessing** - Matching input format to what the pre-trained model expects
3. **Building the model** - Frozen backbone + custom classification head
4. **Training Phase 1** - Training only the classification head (backbone frozen)
5. **Evaluation** - Confusion matrix and prediction visualization
6. **Fine-tuning (Bonus)** - Unfreezing backbone layers for further improvement

### Why Transfer Learning?

Training a CNN from scratch requires massive datasets and compute. Instead, we use **EfficientNetB0** - a model pre-trained on ImageNet (14M images, 1000 classes). It has already learned to detect edges, textures, shapes, and complex patterns. We simply replace its classification head with our own 3-class head (A, B, C) and train that small portion.

### Key Results

| Approach | Test Accuracy |
| --- | --- |
| Baseline CNN (Notebook 1) | ~33-98% (unstable) |
| CNN + Augmentation (Notebook 2) | ~62% |
| **Transfer Learning (this notebook)** | **~99%** |

### Prerequisites

- Run **Notebook 1** first to generate processed datasets
- Pre-trained weights will be downloaded automatically from Keras

---

---
## 1.&nbsp; Import Libraries and download the images 🛠️

In [ ]:
!pip install gdown

In [ ]:
import os
import shutil
import gdown

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from tensorflow.data import Dataset, AUTOTUNE
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, RandomFlip, RandomRotation, RandomContrast
from tensorflow.keras.callbacks import History, EarlyStopping
from tensorflow.keras.optimizers import AdamW

from sklearn.metrics import confusion_matrix

from numpy.random import Generator
from typing import Tuple, Dict
from tensorflow.data import Dataset

In [ ]:
#@title Defining Data Manipulation Functions (from previous notebook)
def unzip_and_move_to_grouped_dir(share_link: str) -> None:
    """
    Unzips zip file of grouped photos and moves everything under the root directory into a content/data/grouped/ folder.

    Args:
        share_link (str): Google Drive share link
    """
    # Setup paths
    file_id = share_link.split('/')[-2]
    zip_path = "/content/grouped_data.zip"
    temp_dir = "/content/temporary"
    dest_dir = "/content/data/grouped"

    os.makedirs(dest_dir, exist_ok=True)

    # Download and unzip
    gdown.download(id=file_id, output=zip_path)
    !unzip -q {zip_path} -d {temp_dir}

    # Remove any __MACOSX folder
    macosx_path = os.path.join(temp_dir, "__MACOSX")
    if os.path.exists(macosx_path):
        shutil.rmtree(macosx_path)

    # Move contents of zip (but not the root folder) to data/grouped/
    # Look inside the temporary folder to find the unknown root (e.g., 'grouped-dataset')
    for root_name in os.listdir(temp_dir):
        if root_name.startswith('.'): continue # skip hidden files
        root_path = os.path.join(temp_dir, root_name)
        if not os.path.isdir(root_path): continue

        # Look at the actual groups (ben, rockwell, ...)
        for group_name in os.listdir(root_path):
            if group_name.startswith('.'): continue # skip hidden files
            source_group = os.path.join(root_path, group_name)
            dest_group = os.path.join(dest_dir, group_name)

            # If the group folder already exists in destination overwrite it but notify user
            if os.path.exists(dest_group):
                print(f"Overwriting {group_name} with new data")
                shutil.rmtree(dest_group)
            shutil.move(source_group, dest_group)

    # Cleanup
    shutil.rmtree(temp_dir)
    os.remove(zip_path)

def _monte_carlo_split(
    groups_dict: dict[str, int],
    target_samples: int,
    seed: int,
    iterations: int = 1000
) -> Tuple[list[str], list[str]]:
    """
    Helper function for group_aware_split. Method for randomized splitting of variously sized groups according to a target sample size.
    We might want x% of the data in the test set, but if groups are varying sizes we can't rely on x% of groups equaling x% of the total samples.
    The monte carlo method repeatedly shuffles the groups and tries to split them as close to the target ratio as possible, and then returns the closest split it found.

    Args:
        groups_dict (dict): Mapping of group names to their total number of images.
        target_samples (int): The number of samples desired for the split. Algorithm will get as close as it can, without spreading any groups across splits.
        seed (int): Random seed for reproducibility.
        iterations (int): Number of random shuffles to try.

    Returns:
       tuple[list[str], list[str]]: List of group names in the split and list of remaining group names.
    """
    # Don't split if split size is 0
    if target_samples == 0:
        return [], list(groups_dict.keys())

    group_names: list[str] = list(groups_dict.keys())

    # Initialize variables to store whatever split is the best so far
    # Whevever the algorithm finds something better, it will overwrite what is stored here
    best_error: float = float('inf')
    best_split: list[str] = []
    best_remaining: list[str] = []

    for i in range(iterations):
        # Randomly shuffle groups
        iteration_rng: Generator = np.random.default_rng(seed + i)
        shuffled_groups: list[str] = iteration_rng.permutation(group_names).tolist()

        # Calculate the cumulative sum of samples for this specific shuffle
        cum_sums: np.ndarray = np.cumsum([groups_dict[g] for g in shuffled_groups])

        # Find which index gets us closest to our target sample count
        errors: np.ndarray = np.abs(cum_sums - target_samples)
        best_idx_for_this_shuffle = np.argmin(errors)

        # If this shuffle produced a more accurate split than previous ones, save it
        if errors[best_idx_for_this_shuffle] < best_error:
            best_error: int = errors[best_idx_for_this_shuffle]
            best_split = shuffled_groups[:best_idx_for_this_shuffle + 1]
            best_remaining = shuffled_groups[best_idx_for_this_shuffle + 1:]

            # Early stopping if we hit it exactly
            if best_error == 0:
                break

    return best_split, best_remaining

def group_aware_split(
    grouped_dir: str,
    raw_dir: str,
    test_size: float = 0.1,
    val_size: float = 0.1,
    test_seed: int = 0,
    val_seed: int = 0,
) -> None:
    """
    Performs group-aware split into training, validation, and testing sets and reorganizes data to set up for tensorflow.
    Creates data/raw/split/label/img file tree using data/grouped/group/label/img file tree.

    Args:
        grouped_dir (str): Directory with grouped images in grouped_dir/group/label/img structure.
        raw_dir (str): Location to save reorganized data in raw_dir/split/label/img structure.
        test_size (float): Percentage of samples to be used for testing.
        val_size (float): Percentage of samples to be used for validation.
        test_seed (int): Random seed to fix testing set group selection.
        val_seed (int): Random seed to fix validation set group selection (does not fix if test_seed changed).
    """
    # Identify groups and count how many samples they have
    groups_dict: dict[str, int] = {}
    for g in os.listdir(grouped_dir):
        if g.startswith('.'): continue # ignore hidden files
        group_path = os.path.join(grouped_dir, g)
        if os.path.isdir(group_path): # ignore non-folder files
            # Count all images inside all label folders for this group
            img_count: int = sum(
                len([f for f in os.listdir(os.path.join(group_path, label)) if not f.startswith('.')])
                for label in os.listdir(group_path)
                if not label.startswith('.') and os.path.isdir(os.path.join(group_path, label))
            )
            # Keep track of group names and their number of samples
            groups_dict[g] = img_count

    # Calculate how many samples should be in validation and testing splits
    total_samples: int = sum(groups_dict[g] for g in groups_dict.keys())
    test_samples: int = int(total_samples * test_size)
    val_samples: int = int(total_samples * val_size)

    # Use monte carlo splitting to split by group while approximating split sample size
    test_groups, remaining_groups = _monte_carlo_split(groups_dict, test_samples, test_seed)
    remaining_groups_dict: dict[str, int] = {g: groups_dict[g] for g in remaining_groups}
    val_groups, train_groups = _monte_carlo_split(remaining_groups_dict, val_samples, val_seed)

    # Pair split names (for dir names) and groups lists
    splits: Dict[str, list[str]] = {
        'test': test_groups,
        'val': val_groups,
        'train': train_groups
    }

    # Remove any existing train-test-val file tree
    # (we can't rely on overwriting if, for instance, we changed to test_size=0)
    if os.path.exists(raw_dir):
        shutil.rmtree(raw_dir)

    # Copy imgs from grouped/group/label/img tree to raw/split/label/img tree
    for split_name, group_list in splits.items():
        for group in group_list:
            group_path = os.path.join(grouped_dir, group) # data/grouped/group

            for label in os.listdir(group_path):
                source_path = os.path.join(group_path, label) # data/grouped/group/label
                # Skip any hidden folders or non-dir files
                if label.startswith('.') or not os.path.isdir(source_path):
                    continue
                dest_label: str = label.upper() # make all labels uppercase in new file tree
                dest_path = os.path.join(raw_dir, split_name, dest_label) # data/raw/split/label
                os.makedirs(dest_path, exist_ok=True)

                for image_file in os.listdir(source_path):
                    if image_file.startswith('.'): continue # skip any hidden files

                    # Get absolute paths to ensure symlinks don't break
                    src_img = os.path.abspath(os.path.join(source_path, image_file)) # .../data/grouped/group/label/img
                    dest_img = os.path.abspath(os.path.join(dest_path, f"{group}_{dest_label}_{image_file}")) # .../data/raw/split/label/img

                    # Skip hassle of symlinks with windows and just duplicate imgs instead
                    if os.name == 'nt':
                        shutil.copy2(src_img, dest_img)
                    else:
                        # Use symlinks to only store images once but in two locations
                        os.symlink(src_img, dest_img)

def process_raw_imgs(
    raw_dir: str,
    processed_dir: str
) -> None:
    """
    Converts images into tensorflow.data.Dataset objects and saves them for fast loading.

    Args:
        raw_dir (str): Path to directory with raw images in raw_dir/split/label/img structure.
        processed_dir (str): Directory where processed data will be saved.
    """
    # Identify splits present in raw directory
    splits: list[str] = [
        s for s in os.listdir(raw_dir)
        if os.path.isdir(os.path.join(raw_dir, s))
    ]

    # Remove any existing processed file tree
    # (we can't rely on overwriting if, for instance, we changed to test_size=0)
    if os.path.exists(processed_dir):
        shutil.rmtree(processed_dir)

    for split in splits:
        # We only want the training set to shuffle
        # (shuffle=True changes the dataset object to reshuffle each time it is referenced which causes problems for things like confusion matrices later on)
        should_shuffle: bool = (split == "train")
        # Auto label images and convert to tensorflow.data.Dataset obejct
        ds: Dataset = image_dataset_from_directory(
            directory=os.path.join(raw_dir, split),
            labels='inferred',
            label_mode='int',
            image_size=(256, 256),
            shuffle=should_shuffle,
            pad_to_aspect_ratio=True,
            batch_size=None
        )
        # Save dataset
        dest_path = os.path.join(processed_dir, split)
        ds.save(dest_path)

### 1.1 Loading Data into Colab and Processing it into Tensorflow Dataset Objects ☁

In colab we have to rerun all of this because the processed Datasets were deleted with the colab runtime. When you're working locally you won't need to repeat this every time.

In [ ]:
student_photos_drive_link: str = "https://drive.google.com/file/d/1R8Bzg47tP9YWWM7QSCGLbt2EVVdBbJLb/view?usp=sharing"

In [ ]:
instructor_photos_drive_link: str = "https://drive.google.com/file/d/1CWvyWJyz-EYHyAG86r8hLBVcG7fez3-k/view?usp=drive_link"

GROUPED_DIR: str = "/content/data/grouped"
RAW_DIR: str = "/content/data/raw"
PROCESSED_DIR: str = "/content/data/processed"

# Clear any existing data
if os.path.exists(GROUPED_DIR): shutil.rmtree(GROUPED_DIR)

# Unzip and move student and instructor photos
unzip_and_move_to_grouped_dir(instructor_photos_drive_link)
unzip_and_move_to_grouped_dir(student_photos_drive_link)

# Perform group-aware train-test-val split
group_aware_split(
    grouped_dir=GROUPED_DIR,
    raw_dir=RAW_DIR,
    test_size=0.125,
    val_size=0.125,
    test_seed=42,
    val_seed=42,
)

# Process images into 256x256x3 tensors
process_raw_imgs(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR
)

### 1.2 Loading, Batching, and Prefetching Datasets 🌩

This bit you will need to rerun even when working locally.

In [ ]:
#@title Defining Dataset Loading Function (from previous notebook)
def load_datasets(
    processed_dir: str,
    batch_size: int = 32
) -> Dict[str, Dataset]:
    """
    Loads, batches, and prefetches processed data. Returns in dictionary with tf.data.Dataset objects for each split.

    Args:
        processed_dir (str): Path to directory with dir/split/Dataset structure
        batch_size (int): Batch size

    Returns:
        dict: mapping of split names to their respective tf.data.Dataset objects

    Examples:
    >>> datasets = load_datasets("../data/processed/", batch_size=32)
    >>> model.fit(
    ...     datasets['train'],
    ...     validation_data=datasets['val'],
    ...     epochs=10
    ... )
    """
    # Identify splits present in processed directory
    splits: list[str] = [
        s for s in os.listdir(processed_dir)
        if os.path.isdir(os.path.join(processed_dir, s))
    ]

    # Initialize autotune for prefetching
    autotune = tf.data.AUTOTUNE

    # Initialize dict that will be returned
    datasets: dict[str, Dataset] = {}

    for split in splits:
        # Load each dataset
        path = os.path.join(processed_dir, split)
        ds: Dataset = Dataset.load(path)
        # Batch, cache, and prefetch for optimal performance
        datasets[split]  = (
            ds
            .batch(batch_size)
            .cache()
            .prefetch(buffer_size=autotune)
        )

    return datasets

In [ ]:
BATCH_SIZE: int = 32

# Load saved Dataset objects, batching and prefetching them
datasets: Dict[str, Dataset] = load_datasets(
    processed_dir=PROCESSED_DIR,
    batch_size=BATCH_SIZE
)

---
## 2.&nbsp; Preprocessing ⚡

Pretrained models were trained on data that had already undergone some preprocessing. We need to preprocess our images the same way so that they look like the kind of thing the model was trained on.

For example, if the model is expecting images that haven't been scaled, we shouldn't scale them first or to the model they'll all look extremely dark.

But how do we know what kind of preprocessing the model needs? Lucky for us, for each pretrained model keras offers, they also have a module that contains all the preprocessing it requires.

We just have to import these modules and convert the preprocessing function into a proper keras layer so that we can place it in the pipeline. We can do this conversion with a lambda layer:

In [ ]:
from tensorflow.keras.applications import efficientnet
from tensorflow.keras.layers import Lambda

preprocessor: Lambda = Lambda(efficientnet.preprocess_input, name='preprocessor')

---
## 3.&nbsp; Initialising the Backbone 🧠

As you should have read on the LMS, when we import a pre-trained classifier that we want to customize for a new classification task it wasn't already trained for, we remove what is called the "classification head" and replace it with a new one.

Since the EfficientNet models were not trained to classify German sign language, we need to remove the pre-trained model's classification head and just keep the feature extractor, which we call the "backbone."

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

# feature_extractor = EfficientNetB0(
#     input_shape=(256, 256, 3),
#     include_top=False, # <--- removing the classifier so we can train it on a new task
#     weights='imagenet',
#     name='backbone'
# )

NOTE: If you are getting a **URL fetch failure**, it's a problem with the API permissions not your code. As an alternative, you can download the .h5 file containing the model weights and provide a filepath to the `weights` parameter. Do not download these files from untrusted sources, your instructor should be able to share the .h5 files with you for each of the pretrained models available on keras.

In [ ]:
feature_extractor: EfficientNetB0 = EfficientNetB0(
    input_shape=(256, 256, 3),
    include_top=False,
    weights='/content/EfficientNetB0_notop.h5', # <--- providing a filepath instead
    name='backbone'
)

Since we'll be training a new head, we need to freeze the weights of the backbone, at least for now.

When we begin training the new head, it will start off with randomly initialized weights, so it will perform very poorly. If we don't freeze the weights in our pretrained model, backpropogation will send that strong error signal backwards through entire network and "shatter" the pretrained backbone, scrambling all it's carefully tuned weights. Even if we want to fine-tune the backbone later, we need to freeze it for now to avoid this effect, which we call "gradient shock."

In [ ]:
# Freeze pre-trained weights
feature_extractor.trainable = False

---
## 4.&nbsp; Initialising the Head 👤

Our classification head should generally be a fairly shallow dense network. It's job is essentially just to learn how to bring together the tools of the pretrained model to apply them to this specific task.

For image classification tasks, the pre-trained backbones will almost always output a stack of feature maps. In other words, **they will not have flattened the activations yet so they can't plug directly into a dense layer**.

A simple `Flatten()` layer could do the trick so we can feed those outputs into a dense network. However, the way these models are trained often lends itself better to an alternative method for going from this block of feature maps to a list of input nodes: Global Pooling.

### 4.1 Global Pooling 🌎

We've seen Max Pooling before as a method for reducing the resolution of feature maps while retaining their most notable information. With Global Pooling, we take the entire feature map and summarize it with a single number.

The most common method of global pooling is `GlobalAveragePooling()` which simply averages all the pixels in each feature map. The result is that we convert our 3D block of feature maps into a 1D list of averages. Now that the data is in a list, it can be fed into a dense network.

The biggest benefit of this method is that it vastly reduces the number of parameters in our first dense layer. If each of the feature maps in the final layer of the backbone were even as small as 7x7 pixels, using global pooling instead of flattening will reduce the number of parameters in that layer by over 98%, significantly reducing the risk of overfitting.

(Note: the pre-trained model must be fit with a head that uses GAP in order for this to work properly. Most modern models use GAP, but older models like VGG16 rely on spatial specificity being retained, so they will suffer from global pooling and should be used with `Flatten()` instead)

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling2D

classifier: Sequential = Sequential([
    # Pooling feature maps
    GlobalAveragePooling2D(),

    # Hidden dense layer
    Dense(64, activation='relu'),

    # Output layer
    Dense(3, activation='softmax')
], name='classification_head')

You may want to experiment with the architecture of your classifier, changing the number of nodes and layers, incorporating dropout layers, and so on.

## 5.&nbsp; Bringing it all Together ⛓

Now that we have out preprocesor, feature extractor, and classifier, we're ready to combine them into the full model!

In [ ]:
RANDOM_SEED: int = 42

data_augmentor: Sequential = Sequential([
    RandomFlip("horizontal", seed=RANDOM_SEED),
    RandomRotation(0.9, seed=RANDOM_SEED),
    RandomContrast(factor=1, seed=RANDOM_SEED)
], name='image_augmentor')

model: Sequential = Sequential([
    Input(shape=(256, 256, 3)),
    data_augmentor,
    preprocessor,
    feature_extractor,
    classifier
], name='full_pipeline')

model.summary()

We can see that most of the parameters in the model are not trainable. Those are the ones in the pretrained backbone that we froze.

Now it's time to fit the model!

In [ ]:
# Initialise optimizer with weight decay
optimizer: AdamW = AdamW(learning_rate=1e-3, weight_decay=1e-4)

# Compile model with optimizer
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Initialise early stopping
es: EarlyStopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train
history: History = model.fit(
    datasets['train'],
    epochs=10,
    validation_data=datasets['val'],
    callbacks=[es]
)

In [ ]:
acc: list[float] = history.history['accuracy']
val_acc: list[float] = history.history['val_accuracy']
loss: list[float] = history.history['loss']
val_loss: list[float] = history.history['val_loss']
epochs_range: range = range(len(acc))

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Train')
plt.plot(epochs_range, val_acc, label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Train')
plt.plot(epochs_range, val_loss, label='Val')
plt.title('Loss')
plt.legend()
plt.show()

---
## 6.&nbsp; Evaluating the model 🔎

Let's have a look at the test set and see where our model suceeds and fails.

In [ ]:
model.evaluate(datasets['test'])

In [ ]:
class_names: list[str] = ['A', 'B', 'C']

# Collect all images and labels from the test set
all_images: list[tf.Tensor] = []
all_labels: list[tf.Tensor] = []
for images, labels in datasets['test']:
    all_images.append(images)
    all_labels.append(labels)

all_images: tf.Tensor = tf.concat(all_images, axis=0)
all_labels: tf.Tensor = tf.concat(all_labels, axis=0)

preds: np.ndarray = model.predict(all_images)
pred_labels: np.ndarray = np.argmax(preds, axis=1)

In [ ]:
num_images: int = 30

# Calculate the number of rows and columns for the subplot grid
n_cols: int = 5  # You can adjust the number of columns as needed
n_rows: int = (num_images + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 2.5, n_rows * 3)) # Adjust figure size based on grid size
for i in range(num_images):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(all_images[i].numpy().astype("uint8"))
    true_label: str = class_names[all_labels[i]]
    predicted_label: str = class_names[pred_labels[i]]
    plt.title(
        f"True: {true_label}\nPred: {predicted_label}",
        color='green' if true_label == predicted_label else 'red')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Create the confusion matrix
cm: np.ndarray = confusion_matrix(all_labels, pred_labels)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

Looking at this performance, you might be able to tell now why transfer learning is standard practice in the industry!

---
## 7.&nbsp; Challenge 🤖

Try using transfer learning on your own image data!

In addition to what is shown here, there is plenty more to experiement with!
1. Try different pre-trained models. You can find the list of models available through keras [here](https://keras.io/api/applications/). Don't forget that different models also require different preprocessors. The pattern of using `preprocessor = Lambda(_________.preprocess_input)` should stay the same though.
    * Remember that if you're getting a URL fetch error when you try to initialise the feature extractor ask your instructor for the .h5 file so you can download and access the weights locally instead of having to fetch them.
2. Try changing your classification head's architecture. Maybe a different number of nodes or a couple dropout layers could improve performance further.
3. Adjust other parts of your pipeline. Your optimiser, your image augmentor, your batch size, there's still a lot to experiment with
5. Try fine-tuning (see below).


---
## [Bonus] Fine-Tuning 🎶

As mentioned on the LMS, Fine-Tuning is when we adjust the weights in the pre-trained portion of the network.

As mentioned above, we only do this after having already trained the classification head with the weights frozen. Now that we've done so, we can unfreeze the backbone and train for longer to see if we can get the loss even lower.

When we unfreeze these weights, we have to reinitialise the optimizer and recompile the model in order for the change to take effect.

In [ ]:
# Unfreeze pre-trained weights
feature_extractor.trainable = True

# Re-initialise optimiser
optimizer: AdamW = AdamW(
    learning_rate=1e-5, # <--- decrease the learning rate to reduce the risk of catastrophic forgetting
    weight_decay=1e-5
)

# Recompile
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
    run_eagerly=True # <--- add this line if you get a NotImplementedError without it
)

# Train for longer
continued_history = model.fit(
    datasets['train'],
    epochs=20, # <--- WARNING: each epoch will take significantly longer!
    validation_data=datasets['val'],
    callbacks=[es]
)

In [ ]:
# Concatenating previous training history with fine-tuning history
epochs_before_fine_tuning: int = len(loss)
extended_loss: list[float] = loss + continued_history.history['loss']
extended_val_loss: list[float] = val_loss + continued_history.history['val_loss']
epochs_range: range = range(len(extended_loss))

plt.plot(epochs_range, extended_loss, label='Train')
plt.plot(epochs_range, extended_val_loss, label='Val')
plt.axvline(x=epochs_before_fine_tuning, color='green', linestyle='--', label='fine-tuning begins')
plt.title('Loss')
plt.legend()
plt.show()

Why does the training loss spike so much during fine-tuning but the validation loss doesn't? This is most-likely a result of dropout-like techniques inside the backbone which, like dropout, hurt the model's performance during training but in a way that makes it more robust during testing.

---
## Summary

In this notebook, we:

1. **Leveraged transfer learning** with EfficientNetB0 pre-trained on ImageNet
2. **Froze the backbone** and trained only a lightweight classification head
3. **Achieved ~99% accuracy** - a massive improvement over the baseline CNN (~62%)
4. **Explored fine-tuning** - unfreezing backbone layers for further improvement

### Key Takeaway

Transfer learning is the **industry standard** for image classification tasks with limited data. By reusing features learned on ImageNet (edges, textures, shapes), we avoid training from scratch and achieve near-perfect accuracy with minimal compute.

### Next Step

Proceed to **[Notebook 4: Final Model](04_final_model.ipynb)** for the competition-ready model with advanced fine-tuning, smart augmentation, and comprehensive evaluation.